# AI 2 · 1주차 전처리 실험

조건별로 찍은 패키지 사진에 전처리를 적용하고, **전처리 전/후 OCR 결과를 비교**합니다.
이 노트북을 끝까지 돌리면 1주차 완료 기준인 **"이미지 전처리 샘플"**이 만들어집니다.

**실행 전 확인**
- `data/raw/`에 `제품번호_조건ID_번호.jpg` 형식으로 사진이 올라가 있어야 합니다. (예: `p01_c0_1.jpg`)
- GPU는 없어도 됩니다. 사진 50~80장 기준 15~20분 정도 걸립니다.
- 위에서부터 한 셀씩 차례로 실행하세요. (셀 선택 후 `Shift + Enter`)

## 1. 레포 받아오기

레포 이름을 바꿨다면 `REPO_URL`만 고치면 됩니다.
비공개 레포라면 왼쪽 열쇠 아이콘(보안 비밀)에 `GITHUB_TOKEN`이라는 이름으로 토큰을 저장하세요. 코드에 토큰을 직접 붙여넣지 마세요.

In [ ]:
REPO_URL = "https://github.com/DL-2026F-Team05/package-ocr-preprocessing.git"
OCR_ENGINE = "paddle"   # 설치나 실행 오류가 나면 "easyocr"로 바꾸고 이 셀부터 다시 실행

import os, subprocess, sys

os.chdir("/content")
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")   # 비공개 레포일 때만 필요
except Exception:
    token = None
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if not os.path.exists("package-ocr"):
    if subprocess.run(["git", "clone", "-q", url, "package-ocr"]).returncode != 0:
        raise RuntimeError("레포를 받아오지 못했습니다. REPO_URL과 공개/비공개 설정(토큰)을 확인하세요.")
os.chdir("/content/package-ocr")
subprocess.run(["git", "pull", "-q"])
print("현재 위치:", os.getcwd())

## 2. 패키지 설치

설치가 끝난 뒤 **"세션 다시 시작"** 안내가 뜨면 다시 시작하고, 1번 셀부터 다시 실행하세요.

In [ ]:
pkgs = ["paddlepaddle", "paddleocr"] if OCR_ENGINE == "paddle" else ["easyocr"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
print("설치 완료:", ", ".join(pkgs))

## 3. 사진과 정답 라벨 확인

파일 이름 규칙에 맞지 않는 사진은 실험에서 빠집니다. 건너뛴 파일이 있으면 이름을 고쳐서 다시 올리세요.
아이폰 `.HEIC` 사진은 JPG로 바꿔야 합니다.

In [ ]:
from collections import Counter
from pathlib import Path

sys.path.insert(0, "preprocessing")
import experiment

RAW, GEN, OUT = Path("data/raw"), Path("data/generated"), Path("results/ai2_week1")

raw_images, skipped = experiment.list_images(RAW)
counts = Counter(experiment.parse_name(p)[1] for p in raw_images)
products = sorted({experiment.parse_name(p)[0] for p in raw_images})

print(f"사진 {len(raw_images)}장, 제품 {len(products)}개: {', '.join(products)}")
for cond in sorted(counts):
    print(f"  {cond} {experiment.COND_NAMES.get(cond, '')}: {counts[cond]}장")
if skipped:
    print("\n[건너뜀] 이름 규칙에 맞지 않는 파일:", ", ".join(skipped))

labels = experiment.load_labels("data/labels/labels.csv")
print(f"\n정답 라벨이 있는 제품: {len(labels)}개")
if not labels:
    print("라벨이 없으면 CER 없이 OCR 결과와 비교 이미지만 만듭니다.")

## 4. 저조도(C1)·흔들림(C3) 사진 만들기

제품마다 첫 번째 정상(C0) 사진으로 3단계씩 만듭니다. `data/generated/`에 저장되고 GitHub에는 올라가지 않습니다.

In [ ]:
generated = experiment.generate_from_clean(RAW, GEN, conditions=("c1", "c3"))
print(f"{len(generated)}장 생성:", ", ".join(p.name for p in generated[:6]), "..." if len(generated) > 6 else "")

## 5. OCR 불러오기와 시험 실행

처음 실행할 때 모델을 내려받느라 1~2분 걸립니다. 첫 번째 사진의 인식 결과가 나오면 정상입니다.

In [ ]:
from ocr_eval import OCREngine
from preprocess import load_image

ocr = OCREngine(OCR_ENGINE)
first = raw_images[0] if raw_images else generated[0]
result = ocr(load_image(first))
print(f"{first.name}: 글자 영역 {len(result)}개")
for r in result[:10]:
    print(f"  {r['score']:.2f}  {r['text']}")

## 6. 전처리 전/후 비교 실험

사진마다 **전처리 없음**과 **조건별 전처리**로 OCR을 두 번 돌립니다. 조건별 전처리 조합은 `preprocessing/preprocess.py`의 `RECIPES`에 있습니다.

In [ ]:
df = experiment.run_experiment(raw_images + generated, ocr, labels, OUT)
print("\n저장 위치:", OUT)

## 7. 조건별 요약

- `brightness`(밝기), `sharpness`(선명도), `glare_ratio`(반사광 비율): 조건마다 사진 품질이 숫자로 얼마나 다른지
- `cer_before` → `cer_after`: 전처리 전후 오류율 (라벨이 있을 때만)
- `cer_change`가 음수면 전처리로 좋아진 것, 양수면 오히려 나빠진 것
- 라벨이 없으면 `boxes_before` → `boxes_after`(찾은 글자 영역 수)로 대략 비교할 수 있습니다

In [ ]:
import pandas as pd

summary = pd.read_csv(OUT / "summary.csv")
summary

## 8. 전처리 전/후 비교 이미지 보기

조건별로 한 장씩 보여줍니다. 전체 이미지는 `results/ai2_week1/compare/`에 있습니다.

In [ ]:
import cv2
import matplotlib.pyplot as plt

shown = set()
for path in sorted((OUT / "compare").glob("*.jpg")):
    cond = experiment.parse_name(path)[1]
    if cond in shown:
        continue
    shown.add(cond)
    plt.figure(figsize=(14, 5))
    plt.imshow(cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB))
    plt.title(f"{path.name} ({cond})")
    plt.axis("off")
    plt.show()

## 9. 결과 내려받기

`ai2_week1_results.zip`이 내려받아집니다. 압축을 풀어 GitHub 레포의 `results/ai2_week1/` 폴더에 **Add file → Upload files**로 올리면 됩니다.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/ai2_week1_results", "zip", OUT)
files.download("/content/ai2_week1_results.zip")

## 다음에 할 일

1. `summary.md`의 **오인식이 심한 사진** 표에서 `cause`(원인) 칸을 사진을 보며 채우기. 예) "반사광이 날짜 숫자를 덮음", "점 글씨가 끊겨 0을 O로 인식"
2. 전처리로 **나빠진** 조건이 있으면 그것도 결과입니다. 왜 나빠졌는지 한 줄 적어두기
3. 비교 이미지 2~3장을 골라 팀 단톡방에 공유